# Custom middleware
When no prebuilt fits, write your own: subclass AgentMiddleware and implement the hooks you care about, or use the lightweight decorator form for a single hook.

## Concept
Node-style hooks (before_model, after_model, …) receive state and runtime and return a state update dict (or None). Wrap-style hooks (wrap_model_call, wrap_tool_call) receive a request plus a handler callable they invoke — letting you retry, modify, or short-circuit.

In [2]:
from typing import Any, Callable
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, ModelRequest, ModelResponse,
)
from langchain.agents.middleware.types import ExtendedModelResponse
from langchain_core.messages import AIMessage
from langgraph.runtime import Runtime

class LoggingMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState[Any], runtime: Runtime[None]) -> dict[str, Any] | None:
        print(f"about to call model with {len(state['messages'])} messages")
        return None
    
    def wrap_model_call(self, request: ModelRequest[None], handler: Callable[[ModelRequest[None]], ModelResponse[Any]]) -> ModelResponse[Any] | AIMessage | ExtendedModelResponse[Any]:
        for attempt in range(3):
            try:
                return handler(request)
            except Exception:
                if(attempt == 2):
                    raise


model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        LoggingMiddleware()
    ]
)

response = agent.invoke({"messages": [{"role": "user",
        "content": "search for a movie which is scifi and latest."}]})
print(response["messages"][-1].content)

about to call model with 1 messages
Here are a few very recent (2023‑2025) sci‑fi movies that have generated buzz and are easy to find on streaming platforms or in theatres. If you’re looking for something else— a specific sub‑genre (e.g., cyber‑punk, space‑opera), a particular tone (light‑hearted vs. mind‑bending), or a title from a specific year—just let me know and I can fine‑tune the list!

---

## 2025

| Title | Synopsis | Why it stands out | Release & Availability |
|-------|----------|------------------|------------------------|
| **“The Ark”** (Netflix) | After Earth’s climate collapses, a generation ship awakens early and must confront an unknown alien signal. | Combines high‑concept astrophysics with tight character drama; superb visual effects for a streaming budget. | Premiered Jan 2025 on Netflix (worldwide). |
| **“Chronicle of the Rift”** (Warner Bros.) | A physicist discovers a pocket dimension that lets humanity travel instantly across the galaxy, but the tech trigger

## Decorator form: one function == one hook.

In [4]:
from langchain.agents.middleware import before_model

@before_model
def trim_history(state: AgentState, runtime: Runtime) -> dict | None:
    msgs = state["messages"]
    print(msgs)
    if len(msgs) > 50:
        return {"messages": msgs[-50:]}
    return None

agent2 = create_agent(
    model=model,
    tools=[],
    middleware=[trim_history]
)

response = agent2.invoke({"messages": [{"role": "user",
        "content": "search for a movie which is scifi and latest."}]})
print(response["messages"][-1].content)

[HumanMessage(content='search for a movie which is scifi and latest.', additional_kwargs={}, response_metadata={}, id='8dedeed8-3d45-4750-b83e-d37b25acf32a')]
Below is a quick “cheat‑sheet” of the most notable **science‑fiction films that have hit (or are about to hit) the market in the last 2‑3 years**. I’ve grouped them by release year, added a one‑sentence hook, and noted where you can watch them today (or when they’ll arrive on streaming). If you’re looking for something even newer—titles slated for late 2025 or 2026—those are listed at the bottom with expected release windows.

---

## 🎬 2023‑2024 – Recent Sci‑Fi Releases (already out)

| Title | Year | Director | Quick Hook | Where to Stream (US) |
|-------|------|-----------|------------|----------------------|
| **The Creator** | 2023 | Gareth Edwards | A post‑apocalyptic war between humans and AI unfolds when a neutral soldier is sent to negotiate with a mysterious AI‑driven “Creator.” | HBO Max (rent/Buy) |
| **M3GAN** | 2023